In [1]:
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.4 MB/s eta 0:00:00a 0:00:01


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch_geometric.datasets import Coauthor
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv

from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score

device = torch.device("cpu")

print(device)

cpu


In [3]:
dataset = Coauthor(
    root="data/CS",
    name="CS"
)

data = dataset[0]

print(data)
print("Nodes:", data.num_nodes)
print("Edges:", data.edge_index.shape[1])
print("Features:", data.num_features)

Processing...
Done!


Data(x=[18333, 6805], edge_index=[2, 163788], y=[18333])
Nodes: 18333
Edges: 163788
Features: 6805


In [4]:
transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.10,
    is_undirected=True,
    add_negative_train_samples=True
)

train_data, val_data, test_data = transform(data)

train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

print(train_data)

Data(x=[18333, 6805], edge_index=[2, 139222], y=[18333], edge_label=[139222], edge_label_index=[2, 139222])


In [5]:
class TeacherGNN(nn.Module):

    def __init__(self, in_channels, hidden=128):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden, hidden)

    def encode(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x

    def decode(self, z, edge_label_index):

        src, dst = edge_label_index

        return (
            z[src] * z[dst]
        ).sum(dim=-1)

In [6]:
class SEN(nn.Module):

    def __init__(self, in_dim, hidden=128):
        super().__init__()

        self.fc1 = nn.Linear(
            in_dim,
            hidden
        )

        self.fc2 = nn.Linear(
            hidden,
            hidden
        )

    def forward(self, x):

        x = F.relu(
            self.fc1(x)
        )

        x = self.fc2(x)

        return x

In [7]:
class StudentMLP(nn.Module):

    def __init__(self, in_dim, hidden=128):
        super().__init__()

        self.fc1 = nn.Linear(
            in_dim,
            hidden
        )

        self.fc2 = nn.Linear(
            hidden,
            hidden
        )

    def encode(self, x):

        x = self.fc1(x)

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.fc2(x)

        return x

    def decode(self, z, edge_label_index):

        src, dst = edge_label_index

        return (
            z[src] * z[dst]
        ).sum(dim=-1)

In [8]:
teacher = TeacherGNN(
    dataset.num_features,
    128
).to(device)

student = StudentMLP(
    dataset.num_features,
    128
).to(device)

sen = SEN(
    dataset.num_features,
    128
).to(device)

In [9]:
optimizer_teacher = optim.AdamW(
    teacher.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

optimizer_student = optim.AdamW(
    list(student.parameters()) +
    list(sen.parameters()),
    lr=0.001,
    weight_decay=5e-4
)

In [10]:
def train_teacher():

    teacher.train()

    optimizer_teacher.zero_grad()

    z = teacher.encode(
        train_data.x,
        train_data.edge_index
    )

    pred = teacher.decode(
        z,
        train_data.edge_label_index
    )

    loss = F.binary_cross_entropy_with_logits(
        pred,
        train_data.edge_label.float()
    )

    loss.backward()

    optimizer_teacher.step()

    return loss.item()

In [11]:
for epoch in range(1,251):

    loss = train_teacher()

    if epoch % 10 == 0:

        print(
            f"Teacher Epoch {epoch} Loss {loss:.4f}"
        )

Teacher Epoch 10 Loss 0.4953
Teacher Epoch 20 Loss 0.4602
Teacher Epoch 30 Loss 0.4384
Teacher Epoch 40 Loss 0.4178
Teacher Epoch 50 Loss 0.3985
Teacher Epoch 60 Loss 0.3798
Teacher Epoch 70 Loss 0.3576
Teacher Epoch 80 Loss 0.3358
Teacher Epoch 90 Loss 0.3147
Teacher Epoch 100 Loss 0.2935
Teacher Epoch 110 Loss 0.2712
Teacher Epoch 120 Loss 0.2507
Teacher Epoch 130 Loss 0.2303
Teacher Epoch 140 Loss 0.2116
Teacher Epoch 150 Loss 0.1959
Teacher Epoch 160 Loss 0.1791
Teacher Epoch 170 Loss 0.1664
Teacher Epoch 180 Loss 0.1510
Teacher Epoch 190 Loss 0.1417
Teacher Epoch 200 Loss 0.1303
Teacher Epoch 210 Loss 0.1197
Teacher Epoch 220 Loss 0.1130
Teacher Epoch 230 Loss 0.1043
Teacher Epoch 240 Loss 0.0970
Teacher Epoch 250 Loss 0.0933


In [12]:
alpha = 0.1
beta = 0.7
gamma = 0.2

temperature = 3.0

In [13]:
def train_student():

    student.train()
    sen.train()

    optimizer_student.zero_grad()

    with torch.no_grad():

        teacher_z = teacher.encode(
            train_data.x,
            train_data.edge_index
        )

    student_z = student.encode(
        train_data.x
    )

    structural_z = sen(
        train_data.x
    )

    enhanced_z = (
        0.5 * student_z +
        0.5 * structural_z
    )

    teacher_out = teacher.decode(
        teacher_z,
        train_data.edge_label_index
    )

    student_out = student.decode(
        enhanced_z,
        train_data.edge_label_index
    )

    sup_loss = F.binary_cross_entropy_with_logits(
        student_out,
        train_data.edge_label.float()
    )

    kd_loss = F.mse_loss(
        torch.sigmoid(
            student_out / temperature
        ),
        torch.sigmoid(
            teacher_out / temperature
        )
    )

    structure_loss = F.mse_loss(
        enhanced_z,
        teacher_z
    )

    loss = (
        alpha * sup_loss +
        beta * kd_loss +
        gamma * structure_loss
    )

    loss.backward()

    optimizer_student.step()

    return loss.item()

In [14]:
for epoch in range(1,401):

    loss = train_student()

    if epoch % 10 == 0:

        print(
            f"Student Epoch {epoch} Loss {loss:.4f}"
        )

Student Epoch 10 Loss 0.1815
Student Epoch 20 Loss 0.1567
Student Epoch 30 Loss 0.1424
Student Epoch 40 Loss 0.1323
Student Epoch 50 Loss 0.1216
Student Epoch 60 Loss 0.1132
Student Epoch 70 Loss 0.1048
Student Epoch 80 Loss 0.0957
Student Epoch 90 Loss 0.0884
Student Epoch 100 Loss 0.0819
Student Epoch 110 Loss 0.0748
Student Epoch 120 Loss 0.0693
Student Epoch 130 Loss 0.0644
Student Epoch 140 Loss 0.0602
Student Epoch 150 Loss 0.0570
Student Epoch 160 Loss 0.0541
Student Epoch 170 Loss 0.0515
Student Epoch 180 Loss 0.0491
Student Epoch 190 Loss 0.0478
Student Epoch 200 Loss 0.0458
Student Epoch 210 Loss 0.0448
Student Epoch 220 Loss 0.0437
Student Epoch 230 Loss 0.0422
Student Epoch 240 Loss 0.0416
Student Epoch 250 Loss 0.0407
Student Epoch 260 Loss 0.0397
Student Epoch 270 Loss 0.0391
Student Epoch 280 Loss 0.0385
Student Epoch 290 Loss 0.0379
Student Epoch 300 Loss 0.0375
Student Epoch 310 Loss 0.0370
Student Epoch 320 Loss 0.0364
Student Epoch 330 Loss 0.0360
Student Epoch 340 L

In [18]:
def hits_at_k(pos_pred, neg_pred, k):

    pos_pred = pos_pred.cpu()
    neg_pred = neg_pred.cpu()

    scores = torch.cat([pos_pred, neg_pred])

    labels = torch.cat([
        torch.ones(len(pos_pred)),
        torch.zeros(len(neg_pred))
    ])

    _, indices = torch.sort(
        scores,
        descending=True
    )

    labels = labels[indices]

    topk = labels[:k]

    return topk.sum().item() / max(
        1,
        len(pos_pred)
    )

In [19]:
@torch.no_grad()
def evaluate(split):

    student.eval()
    sen.eval()

    z = student.encode(
        split.x
    )

    structural_z = sen(
        split.x
    )

    z = (
        0.5 * z +
        0.5 * structural_z
    )

    pred = student.decode(
        z,
        split.edge_label_index
    )

    pred = torch.sigmoid(pred)

    label = split.edge_label

    auc = roc_auc_score(
        label.cpu().numpy(),
        pred.cpu().numpy()
    )

    ap = average_precision_score(
        label.cpu().numpy(),
        pred.cpu().numpy()
    )

    pos_pred = pred[
        label == 1
    ]

    neg_pred = pred[
        label == 0
    ]

    hit20 = hits_at_k(
        pos_pred,
        neg_pred,
        20
    )

    hit50 = hits_at_k(
        pos_pred,
        neg_pred,
        50
    )

    return (
        auc,
        ap,
        hit20,
        hit50
    )

In [20]:
auc, ap, hit20, hit50 = evaluate(
    test_data
)

print()

print("========== FINAL RESULT ==========")

print(f"AUC      : {auc:.4f}")
print(f"AP       : {ap:.4f}")

print(f"Hits@20  : {hit20*100:.2f}")
print(f"Hits@50  : {hit50*100:.2f}")


========== FINAL RESULT ==========
AUC      : 0.9143
AP       : 0.9061
Hits@20  : 0.24
Hits@50  : 0.61
